## Load Data and Drop Channel

In [279]:
import numpy as np

data = np.load("Datasets/Main.npz")
X = data["X_raw"]
y = data["y"]  

BAD_CHANNELS = [0,1,2,6,7]
X = np.delete(X, BAD_CHANNELS, axis=1)
print(X.shape)

(570, 3, 1750)


## Filtering

In [280]:
import numpy as np
from brainflow.data_filter import (
    DataFilter,
    FilterTypes,
    DetrendOperations
)

SAMPLING_RATE = 250  
LOW_CUT = 8
HIGH_CUT = 13
FILTER_ORDER = 4

X_filtered = np.copy(X)

for trial in range(X.shape[0]):

    for ch in range(X.shape[1]):

        signal = X_filtered[trial, ch]
        signal = np.ascontiguousarray(
            signal,
            dtype=np.float64
        )

        DataFilter.detrend(
            signal,
            DetrendOperations.CONSTANT.value
        )

        DataFilter.perform_bandpass(
            signal,
            SAMPLING_RATE,
            LOW_CUT,
            HIGH_CUT,
            FILTER_ORDER,
            FilterTypes.BUTTERWORTH_ZERO_PHASE.value,
            0
        )

        DataFilter.remove_environmental_noise(
            signal,
            SAMPLING_RATE,
            1
        )

        X_filtered[trial, ch] = signal

print("Original Shape :", X.shape)
print("Filtered Shape :", X_filtered.shape)

X = X_filtered

Original Shape : (570, 3, 1750)
Filtered Shape : (570, 3, 1750)


## Data Snipping

In [281]:
X_snipped = X[:, :, 500:-125]
X = X_snipped
print("Original Shape :", X.shape)
print("Snipped Shape  :", X_snipped.shape)

Original Shape : (570, 3, 1125)
Snipped Shape  : (570, 3, 1125)


## Noisy Trial Rejection

In [282]:
import numpy as np

def reject_noisy_trials(
    X,
    y,
    max_abs_thresh=250,
    ptp_thresh=500,
    std_z_thresh=3.0,
    flat_std_thresh=1e-6,
    verbose=True
):
    """
    X shape: (trials, channels, samples)
    y shape: (trials,)
    """

    # =========================
    # TRIAL / CHANNEL STATS
    # =========================

    max_abs = np.max(np.abs(X), axis=2)    # (trials, channels)
    ptp = np.ptp(X, axis=2)                # (trials, channels)
    std = np.std(X, axis=2)                # (trials, channels)

    trial_max_abs = np.max(max_abs, axis=1)
    trial_ptp = np.max(ptp, axis=1)
    trial_mean_std = np.mean(std, axis=1)

    median_std = np.median(trial_mean_std)
    mad_std = np.median(np.abs(trial_mean_std - median_std)) + 1e-12
    robust_z_std = 0.6745 * (trial_mean_std - median_std) / mad_std

    # =========================
    # REJECTION REASONS
    # =========================

    reason_max_abs = trial_max_abs > max_abs_thresh
    reason_ptp = trial_ptp > ptp_thresh
    reason_std_z = np.abs(robust_z_std) > std_z_thresh
    reason_flat = np.any(std < flat_std_thresh, axis=1)

    reject_mask = (
        reason_max_abs |
        reason_ptp |
        reason_std_z |
        reason_flat
    )

    keep_mask = ~reject_mask

    X_clean = X[keep_mask]
    y_clean = y[keep_mask]

    # =========================
    # SUMMARY
    # =========================

    if verbose:
        print("=" * 60)
        print("TRIAL REJECTION SUMMARY")
        print("=" * 60)

        print("Original trials :", len(X))
        print("Rejected trials :", int(np.sum(reject_mask)))
        print("Remaining trials:", len(X_clean))

        print("\nRejection reason counts:")
        print("Max abs exceeded :", int(np.sum(reason_max_abs)))
        print("PTP exceeded     :", int(np.sum(reason_ptp)))
        print("STD z-score bad  :", int(np.sum(reason_std_z)))
        print("Flat signal      :", int(np.sum(reason_flat)))

        print("\nRejected trial details:")
        rejected_indices = np.where(reject_mask)[0]

        if len(rejected_indices) == 0:
            print("None")
        else:
            for idx in rejected_indices:
                reasons = []

                if reason_max_abs[idx]:
                    reasons.append(
                        f"max_abs={trial_max_abs[idx]:.2f} > {max_abs_thresh}"
                    )

                if reason_ptp[idx]:
                    reasons.append(
                        f"ptp={trial_ptp[idx]:.2f} > {ptp_thresh}"
                    )

                if reason_std_z[idx]:
                    reasons.append(
                        f"std_z={robust_z_std[idx]:+.2f} > ±{std_z_thresh}"
                    )

                if reason_flat[idx]:
                    flat_channels = np.where(std[idx] < flat_std_thresh)[0]
                    reasons.append(
                        f"flat channels={flat_channels.tolist()}"
                    )

                print(f"Trial {idx}: " + " | ".join(reasons))

        print("\nClass balance before:")
        print(dict(zip(*np.unique(y, return_counts=True))))

        print("\nClass balance after:")
        print(dict(zip(*np.unique(y_clean, return_counts=True))))

    return {
        "X_clean": X_clean,
        "y_clean": y_clean,
        "keep_mask": keep_mask,
        "reject_mask": reject_mask,

        "reasons": {
            "max_abs": reason_max_abs,
            "ptp": reason_ptp,
            "std_z": reason_std_z,
            "flat": reason_flat,
        },

        "stats": {
            "trial_max_abs": trial_max_abs,
            "trial_ptp": trial_ptp,
            "trial_mean_std": trial_mean_std,
            "robust_z_std": robust_z_std,
        }
    }


result = reject_noisy_trials(
    X,
    y,
    max_abs_thresh=100,
    ptp_thresh=175,
    std_z_thresh=3.0
)

X = result["X_clean"]
y = result["y_clean"]

keep_mask = result["keep_mask"]
reject_mask = result["reject_mask"]
reasons = result["reasons"]
stats = result["stats"]

TRIAL REJECTION SUMMARY
Original trials : 570
Rejected trials : 41
Remaining trials: 529

Rejection reason counts:
Max abs exceeded : 8
PTP exceeded     : 10
STD z-score bad  : 40
Flat signal      : 0

Rejected trial details:
Trial 6: std_z=+6.35 > ±3.0
Trial 7: max_abs=160.92 > 100 | ptp=320.03 > 175 | std_z=+5.25 > ±3.0
Trial 13: max_abs=1082.23 > 100 | ptp=2121.45 > 175 | std_z=+515.50 > ±3.0
Trial 24: std_z=+5.54 > ±3.0
Trial 33: std_z=+4.14 > ±3.0
Trial 41: std_z=+4.22 > ±3.0
Trial 51: std_z=+3.10 > ±3.0
Trial 52: std_z=+6.71 > ±3.0
Trial 69: std_z=+4.79 > ±3.0
Trial 70: std_z=+3.37 > ±3.0
Trial 83: std_z=+7.03 > ±3.0
Trial 86: std_z=+5.90 > ±3.0
Trial 99: std_z=+4.52 > ±3.0
Trial 110: std_z=+5.28 > ±3.0
Trial 135: std_z=+5.32 > ±3.0
Trial 136: max_abs=1071.43 > 100 | ptp=2114.29 > 175 | std_z=+515.72 > ±3.0
Trial 140: std_z=+5.13 > ±3.0
Trial 156: std_z=+8.37 > ±3.0
Trial 168: std_z=+8.18 > ±3.0
Trial 225: std_z=+4.57 > ±3.0
Trial 244: std_z=+8.97 > ±3.0
Trial 264: std_z=+10.06 >

## Train Test Val Split

In [283]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,   # 70% train
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,   # 15% val, 15% test
    random_state=42,
    stratify=y_temp
)

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)
print("X_test shape :", X_test.shape)

print()

print("y_train shape:", y_train.shape)
print("y_val shape  :", y_val.shape)
print("y_test shape :", y_test.shape)

print("\nTrain class balance:")
print(np.unique(y_train, return_counts=True))

print("\nValidation class balance:")
print(np.unique(y_val, return_counts=True))

print("\nTest class balance:")
print(np.unique(y_test, return_counts=True))

X_train shape: (370, 3, 1125)
X_val shape  : (79, 3, 1125)
X_test shape : (80, 3, 1125)

y_train shape: (370,)
y_val shape  : (79,)
y_test shape : (80,)

Train class balance:
(array([1, 2]), array([187, 183]))

Validation class balance:
(array([1, 2]), array([40, 39]))

Test class balance:
(array([1, 2]), array([40, 40]))


## Data Prep

In [284]:
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train = np.transpose(X_train, (0,2,1))
X_val = np.transpose(X_val, (0,2,1))
X_test = np.transpose(X_test, (0,2,1))

n_trials, n_samples, n_channels = X_train.shape

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_channels)).reshape(n_trials, n_samples, n_channels)
X_val_scaled = scaler.transform(X_val.reshape(-1, n_channels)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_channels)).reshape(X_test.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Change Class Labels to [0,1] from [1,2]
y_train = y_train - 1
y_val = y_val - 1
y_test = y_test - 1

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
X_val = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_val = torch.tensor(y_val, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


Using device: cuda


## LSTM Architecture

In [285]:
import torch
import torch.nn as nn

class BraindanceLSTMClassifier (torch.nn.Module):

    def __init__ (self, input_size, hidden_size, num_layers, num_classes, dropout=0.3, bidirectional=False):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        lstm_output_size = hidden_size * 2 if bidirectional else hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward (self, x):

        lstm_out, (h_n, c_n) = self.lstm(x)

        if self.bidirectional:
            forward_last = h_n[-2, :, :]
            backward_last = h_n[-1, :, :]
            final_hidden = torch.cat((forward_last, backward_last), dim=1)
        else:
            final_hidden = h_n[-1, :, :]

        logits = self.classifier(final_hidden)

        return logits
    
input_size = n_channels
num_classes = 2

model = BraindanceLSTMClassifier(
    input_size=input_size,
    hidden_size=256,
    num_layers=3,
    num_classes=num_classes,
    dropout=0,
    bidirectional=False
).to(device)

## Training Loop

In [286]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ============================================================
# CONFIG
# ============================================================

EPOCHS = 150
PATIENCE = 20

LEARNING_RATE = 1e-2
WEIGHT_DECAY = 0

SAVE_LOWEST_VAL_LOSS_PATH = "best_lstm_lowest_val_loss.pt"
SAVE_HIGHEST_VAL_AUC_PATH = "best_lstm_highest_val_auc.pt"
SAVE_HIGHEST_VAL_ACC_PATH = "best_lstm_highest_val_acc.pt"

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ============================================================
# HISTORY
# ============================================================

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_bal_acc": [],
    "val_bal_acc": [],
    "train_auc": [],
    "val_auc": [],
    "train_f1": [],
    "val_f1": []
}

# ============================================================
# BEST MODEL TRACKERS
# ============================================================

best_val_loss = float("inf")
best_val_auc = -float("inf")
best_val_acc = -float("inf")

best_val_loss_state = None
best_val_auc_state = None
best_val_acc_state = None

best_val_loss_epoch = None
best_val_auc_epoch = None
best_val_acc_epoch = None

epochs_without_improvement = 0


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs_class_1 = []

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            total_loss += loss.item() * xb.size(0)

            all_labels.extend(yb.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())

            if probs.shape[1] == 2:
                all_probs_class_1.extend(probs[:, 1].detach().cpu().numpy())
            else:
                all_probs_class_1.extend([np.nan] * xb.size(0))

    avg_loss = total_loss / len(data_loader.dataset)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs_class_1 = np.array(all_probs_class_1)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs_class_1)
    except ValueError:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "auc": auc,
        "f1": f1,
        "labels": all_labels,
        "preds": all_preds,
        "probs_class_1": all_probs_class_1
    }


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_train_loss = 0.0

    train_labels = []
    train_preds = []
    train_probs_class_1 = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * xb.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        train_labels.extend(yb.detach().cpu().numpy())
        train_preds.extend(preds.detach().cpu().numpy())

        if probs.shape[1] == 2:
            train_probs_class_1.extend(probs[:, 1].detach().cpu().numpy())
        else:
            train_probs_class_1.extend([np.nan] * xb.size(0))

    train_loss = running_train_loss / len(train_loader.dataset)

    train_labels = np.array(train_labels)
    train_preds = np.array(train_preds)
    train_probs_class_1 = np.array(train_probs_class_1)

    train_acc = accuracy_score(train_labels, train_preds)
    train_bal_acc = balanced_accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, zero_division=0)

    try:
        train_auc = roc_auc_score(train_labels, train_probs_class_1)
    except ValueError:
        train_auc = np.nan

    val_results = evaluate_model(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=device
    )

    val_loss = val_results["loss"]
    val_acc = val_results["acc"]
    val_bal_acc = val_results["bal_acc"]
    val_auc = val_results["auc"]
    val_f1 = val_results["f1"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    history["train_bal_acc"].append(train_bal_acc)
    history["val_bal_acc"].append(val_bal_acc)

    history["train_auc"].append(train_auc)
    history["val_auc"].append(val_auc)

    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    improved = False

    # -------------------------
    # Save lowest val loss
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_loss_state = copy.deepcopy(model.state_dict())
        best_val_loss_epoch = epoch

        torch.save(best_val_loss_state, SAVE_LOWEST_VAL_LOSS_PATH)

        improved = True

    # -------------------------
    # Save highest val AUC
    # -------------------------
    if not np.isnan(val_auc) and val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_auc_state = copy.deepcopy(model.state_dict())
        best_val_auc_epoch = epoch

        torch.save(best_val_auc_state, SAVE_HIGHEST_VAL_AUC_PATH)

        improved = True

    # -------------------------
    # Save highest val accuracy
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_acc_state = copy.deepcopy(model.state_dict())
        best_val_acc_epoch = epoch

        torch.save(best_val_acc_state, SAVE_HIGHEST_VAL_ACC_PATH)

        improved = True

    if improved:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Bal Acc: {val_bal_acc:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break


print("\n================ TRAINING COMPLETE ================")
print(f"Best Lowest Val Loss: {best_val_loss:.4f} at epoch {best_val_loss_epoch}")
print(f"Best Highest Val AUC : {best_val_auc:.4f} at epoch {best_val_auc_epoch}")
print(f"Best Highest Val Acc : {best_val_acc:.4f} at epoch {best_val_acc_epoch}")

print("\nSaved models:")
print(SAVE_LOWEST_VAL_LOSS_PATH)
print(SAVE_HIGHEST_VAL_AUC_PATH)
print(SAVE_HIGHEST_VAL_ACC_PATH)


# ============================================================
# TEST BEST MODELS
# ============================================================

def test_saved_state(model, state_dict, test_loader, criterion, device, model_name):
    model.load_state_dict(state_dict)
    model.to(device)

    results = evaluate_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device
    )

    return {
        "name": model_name,
        "loss": results["loss"],
        "acc": results["acc"],
        "bal_acc": results["bal_acc"],
        "auc": results["auc"],
        "f1": results["f1"],
        "labels": results["labels"],
        "preds": results["preds"],
        "probs_class_1": results["probs_class_1"]
    }


test_results = []

test_results.append(
    test_saved_state(
        model=model,
        state_dict=best_val_loss_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Lowest Val Loss"
    )
)

test_results.append(
    test_saved_state(
        model=model,
        state_dict=best_val_auc_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Highest Val AUC"
    )
)

test_results.append(
    test_saved_state(
        model=model,
        state_dict=best_val_acc_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Highest Val Acc"
    )
)


# ============================================================
# PRINT TEST COMPARISON AT TOP
# ============================================================

print("\n================ TEST COMPARISON ================")
print(
    f"{'Model':<20} "
    f"{'Test Loss':<12} "
    f"{'Test Acc':<10} "
    f"{'Test Bal Acc':<14} "
    f"{'Test AUC':<10} "
    f"{'Test F1':<10}"
)

print("-" * 80)

for_result_rows = []

for result in test_results:
    print(
        f"{result['name']:<20} "
        f"{result['loss']:<12.4f} "
        f"{result['acc']:<10.4f} "
        f"{result['bal_acc']:<14.4f} "
        f"{result['auc']:<10.4f} "
        f"{result['f1']:<10.4f}"
    )


# ============================================================
# CLASSIFICATION REPORTS
# ============================================================

for result in test_results:
    print(f"\n================ {result['name']} TEST REPORT ================")
    print(classification_report(
        result["labels"],
        result["preds"],
        zero_division=0
    ))


# ============================================================
# PLOTS: TRAIN LOSS VS VAL LOSS
# ============================================================

epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_loss"], label="Train Loss")
plt.plot(epochs_ran, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train Loss vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# PLOTS: TRAIN ACC VS VAL ACC
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_ran, history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train Accuracy vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# PLOT: VALIDATION AUC
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Validation AUC")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# CONFUSION MATRICES FOR THREE TESTED MODELS
# ============================================================

for result in test_results:
    cm = confusion_matrix(result["labels"], result["preds"])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Class 0", "Class 1"]
    )

    disp.plot(values_format="d")
    plt.title(f"Test Confusion Matrix - {result['name']}")
    plt.grid(False)
    plt.show()

Epoch [001/150] Train Loss: 0.7174 | Val Loss: 0.7040 | Train Acc: 0.4595 | Val Acc: 0.5190 | Val Bal Acc: 0.5250 | Val AUC: 0.5526 | Val F1: 0.6724
Epoch [002/150] Train Loss: 0.7096 | Val Loss: 0.6956 | Train Acc: 0.5081 | Val Acc: 0.4810 | Val Bal Acc: 0.4798 | Val AUC: 0.4821 | Val F1: 0.4225
Epoch [003/150] Train Loss: 0.6947 | Val Loss: 0.6962 | Train Acc: 0.4919 | Val Acc: 0.5063 | Val Bal Acc: 0.5122 | Val AUC: 0.5006 | Val F1: 0.6609
Epoch [004/150] Train Loss: 0.6957 | Val Loss: 0.6935 | Train Acc: 0.4946 | Val Acc: 0.4937 | Val Bal Acc: 0.5000 | Val AUC: 0.4994 | Val F1: 0.6610
Epoch [005/150] Train Loss: 0.6929 | Val Loss: 0.6944 | Train Acc: 0.4946 | Val Acc: 0.4937 | Val Bal Acc: 0.4955 | Val AUC: 0.4878 | Val F1: 0.5556
Epoch [006/150] Train Loss: 0.6997 | Val Loss: 0.6930 | Train Acc: 0.5054 | Val Acc: 0.5063 | Val Bal Acc: 0.5000 | Val AUC: 0.5327 | Val F1: 0.0000
Epoch [007/150] Train Loss: 0.6946 | Val Loss: 0.6926 | Train Acc: 0.5054 | Val Acc: 0.4937 | Val Bal Acc:

KeyboardInterrupt: 